In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
import sys, os

# 本地自写回测引擎 + 本地 DuckDB 数据源（data_source 默认后端离线读 data/ 下 parquet）
# 定位本工程根（含 engine/）：data_source / smalltrader 在 engine/ 下，从 cwd 向上找根目录并入 sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, 'engine')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
sys.path.insert(0, os.path.join(_root, 'engine'))
sys.path.insert(0, _root)
import smalltrader as bigtrader
import data_source  # 注册自动取数：bigtrader.run(data=缺省) 时按 market/instruments 从本地 DuckDB 拉行情

import queries  # 本地 DuckDB：cn_stock_prefactors 等视图（表清单见 queries.TABLES，惰性读 data/ 下 parquet）
con = queries.connect()

START_DATE = '2021-01-01'
END_DATE   = '2026-10-30'
CAPITAL    = 1000000

def initialize(context: bigtrader.IContext):
    # 系统已默认设置交易手续费和滑点，可按需修改
    context.set_commission(bigtrader.PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
    context.holding_days = 5        # 每 5 个交易日调仓一次
    context.target_hold_count = 20   # 每次持有 20 只股票

    # 目标：全市场非 ST、非风险警示股票，按市值升序取最小 N 只
    context.sql = """
    SELECT date, instrument, total_market_cap
    FROM cn_stock_prefactors
    WHERE st_status = 0
      AND is_risk_warning = 0
      AND total_market_cap IS NOT NULL
      AND volume > 0
      AND close > 0
      --AND low < upper_limit   -- 排除当前一字涨停，避免次日开盘买不进
      AND date IS NOT NULL
      AND date BETWEEN ? AND ?
    """
    context.data = con.execute(context.sql, [context.start_date, context.end_date]).df().dropna()
    context.data['date'] = pd.to_datetime(context.data['date']).dt.strftime('%Y-%m-%d')

def befor_trading(context, data):
    pass

def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    # 每 context.holding_days 个交易日调仓一次
    if context.trading_day_index % context.holding_days != 0:
        return

    current_date = data.current_dt.strftime('%Y-%m-%d')
    today = context.data[context.data['date'] == current_date]
    if today.empty:
        return

    # 按市值从小到大排序，取最小的 6 只
    today = today.sort_values(['total_market_cap', 'instrument'], ascending=[True, True])
    selected = today.head(context.target_hold_count)
    target_hold_instruments = set(selected['instrument'])
    target_count = len(selected)

    if target_count == 0:
        return

    # 当前持仓
    current_hold_instruments = set(context.get_account_positions().keys())

    # 卖出不在目标持仓列表中的股票
    for instrument in sorted(current_hold_instruments - target_hold_instruments):
        context.order_target_percent(instrument, 0)

    # 买入/调整到目标持仓列表中的股票（等权，留 10% 现金应对交易成本）
    for instrument in sorted(target_hold_instruments - current_hold_instruments):
        context.order_target_percent(instrument, 0.9 / target_count)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=END_DATE,
    capital_base=CAPITAL,            # 设置初始资金
    initialize=initialize,           # 传入初始化函数
    handle_data=handle_data,         # 传入数据处理函数
    before_trading_start=befor_trading,
    order_price_field_buy='open',
    order_price_field_sell='open',
    render='report'                  # 渲染绩效报告
)


## BigTrader 回测报告

### 绩效摘要

| 指标    | 值 |
|-------|------|
| 累计收益率 | 274.35% |
| 年化收益率 | 26.61% |
| 基准收益率 | -24.34% |
| 阿尔法   | 0.35 |
| 贝塔    | 0.61 |
| 夏普比率  | 0.78 |
| 胜率    | 62.84% |
| 盈亏比   | 1.64 |
| 收益波动率 | 33.66% |
| 信息比率  | 1.0 |
| 最大回撤  | 40.45% |

### 收益走势

| 日期 | 策略收益率 | 基准收益率 | 相对收益率 |
|------|----------|----------|----------|
| 2021-01-04 | 0.0% | 0.0% | 0.0% |
| 2021-01-29 | -14.57% | 1.6% | -16.39% |
| 2021-02-26 | -6.08% | 1.31% | -8.37% |
| 2021-03-31 | -2.68% | -4.16% | -0.13% |
| 2021-04-30 | -3.53% | -2.74% | -2.61% |
| 2021-05-31 | 4.96% | 1.21% | 1.64% |
| 2021-06-30 | 9.78% | -0.83% | 8.37% |
| 2021-07-30 | 7.7% | -8.67% | 15.01% |
| 2021-08-31 | 16.3% | -8.77% | 24.18% |
| 2021-09-30 | 14.67% | -7.62% | 20.77% |
| 2021-10-29 | 12.15% | -6.81% | 17.08% |
| 2021-11-30 | 10.29% | -8.27% | 16.96% |
| 2021-12-31 | 17.2% | -6.21% | 21.44% |
| 2022-01-28 | 11.32% | -13.36% | 24.84% |
| 2022-02-28 | 2.77% | -13.02% | 14.79% |
| 2022-03-31 | -5.03% | -19.84% | 14.72% 

In [2]:
#performance.render()  # 渲染绩效报告并在浏览器中打开